# Schleswig data exploration

Read local data, inspect a small preview, then decide what to prepare later.
**No downloads, source changes or exports.** No file is selected by default.

Use Python 3.12+ and `data/requirements.txt`; setup commands are in the [README](../README.md#optional-data-exploration-notebook).
Select that Python environment as the notebook kernel, edit the next cell, then **Run All**.
Clear outputs before saving for version control.

## 1. Imports and settings

In [51]:
from pathlib import Path
from itertools import islice
import codecs
import csv
import json
import os
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from IPython.display import display

ROOT_OVERRIDE = None  # Absolute repository path if started outside the repository.
SOURCE = r"data\curated\opendata-sh\2026-10-12_Denkmalliste.json"         # e.g. "data/imported/bundesnetzagentur/2026-09-01_Ladesaeulenregister.csv"
FORMAT = "json"      # None = automatic, or "csv", "txt", "json", "parquet".
ENCODING = None      # None = automatic (BOM / UTF-8), or "latin1", "cp1252".
SEPARATOR = None     # None = automatic, or ";", ",", "|", "\t".
HEADER = "auto"       # Auto: no header for TXT, first row for CSV. Override with None or 0.
SKIP_ROWS = 0         # Charging register: 10 introductory lines.
SHOW = 10
LIMIT = 5_000         # Leading records, NOT a random sample. None explicitly loads all.
COLUMNS = None        # Optional CSV column names/positions or Parquet column names.
JSON_PATH = ()      # Inspect first. Then () for root or ("data", "regions").
JSON_MAX_BYTES = 8 * 1024 * 1024

pd.set_option("display.max_rows", SHOW)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

## 2. Choose a file

Paths are relative to the repository root, or absolute. Root discovery checks the working directory and VS Code's notebook path. For other starting directories, use `ROOT_OVERRIDE` or the `SCHLESWIG_MCP_ROOT` environment variable.

Common settings:
- **Charging CSV:** `SEPARATOR=";"`, `SKIP_ROWS=10`, `HEADER=0`.
- **ALKIS TXT:** `SEPARATOR=";"`, `HEADER=None`.
- **Bathing waters:** `SEPARATOR="|"`, `ENCODING="latin1"`, `HEADER=None`.
- **JSON / .JSO:** inspect the structure, then choose `JSON_PATH`.

Reset source-specific settings when switching files. Consult [source documentation](../docs/datasources/) before naming headerless columns.

In [52]:
def repository_root():
    override = ROOT_OVERRIDE or os.environ.get("SCHLESWIG_MCP_ROOT")
    starts = [Path(override)] if override else [
        Path.cwd(), Path(globals().get("__vsc_ipynb_file__", Path.cwd()))
    ]
    for start in starts:
        start = start.expanduser().resolve()
        for parent in [start, *start.parents]:
            manifest = parent / "package.json"
            if manifest.is_file() and json.loads(manifest.read_text(encoding="utf-8")).get("name") == "schleswig-mcp":
                return parent
    raise FileNotFoundError("Set ROOT_OVERRIDE to the repository's absolute path.")


def file_info(path):
    with path.open("rb") as handle:
        prefix = handle.read(65_536)
    kind = FORMAT or {".jso": "json", ".txt": "csv", ".tsv": "csv"}.get(
        path.suffix.lower(), path.suffix.lower().lstrip(".")
    )
    if kind == "txt":
        kind = "csv"
    if kind == "parquet" or prefix.startswith(b"PAR1"):
        with pq.ParquetFile(path) as file:
            return dict(format="parquet", encoding=None, separator=None,
                        records=file.metadata.num_rows)
    encoding = ENCODING
    if encoding is None:
        encoding = next((name for bom, name in [
            (codecs.BOM_UTF32_LE, "utf-32"), (codecs.BOM_UTF32_BE, "utf-32"),
            (codecs.BOM_UTF16_LE, "utf-16"), (codecs.BOM_UTF16_BE, "utf-16"),
            (codecs.BOM_UTF8, "utf-8-sig"),
        ] if prefix.startswith(bom)), "utf-8")
    text = codecs.getincrementaldecoder(encoding)(errors="strict").decode(prefix, final=False)
    if kind not in {"csv", "json"}:
        kind = "json" if text.lstrip().startswith(("{", "[")) else "csv"
    separator = SEPARATOR
    if kind == "csv" and separator is None:
        separator = csv.Sniffer().sniff(
            "\n".join(text.splitlines()[SKIP_ROWS:]), delimiters=";,\t|"
        ).delimiter
    return dict(format=kind, encoding=encoding, separator=separator, records=None)

In [53]:
root = repository_root()
path = (root / Path(SOURCE).expanduser()).resolve() if SOURCE else None
info = file_info(path) if path else None
if path:
    display(pd.Series({"path": str(path), "bytes": path.stat().st_size, **info}))
    if info["format"] != "parquet":
        with path.open(encoding=info["encoding"]) as handle:
            print(handle.read(500))  # Small raw prefix, before any table conversion.
else:
    print("Choose SOURCE in the settings cell to begin.")

path         C:\Users\Niklas\Repositories\schleswig-mcp\data\curated\opendata-sh\2026-10-...
bytes                                                                                7184483
format                                                                                  json
encoding                                                                               utf-8
separator                                                                               None
records                                                                                 None
dtype: object

[
   {
     "Adresse-Lage": "Am Bondenholz 26, 26-30, 30",
     "Beschreibung": "Obsthof Mehrens; 1907; Fabrikation Nordischer Blockhäuser W. Witte; zweigeschossiges Fertig-Holzhaus aus Harzer Fichte auf gemauertem Sockel unter auskragendem Satteldach im Chaletstil; ehemaliges Gärtnerhaus am Seeufer, eingeschossiges Blockhaus auf hohem gemauertem Sockel unter Dach in originaler Biberschwanzdeckung; alter Obstwiesenbestand",
     "Bezeichnung": "Obsthof Mehrens",
     "Kulturdenkmaltyp": "Sachges


## 3. Load a preview

Encoding detection only checks the prefix; configure ambiguous encodings explicitly.
CSV values remain text, preserving leading zeroes and literal `NA`. Header settings are explicit; duplicate/empty column labels are rejected.

JSON is loaded as a document first, never silently flattened. The 8 MiB guard limits document size; `LIMIT` does **not** limit JSON parsing memory. Larger JSON needs a deliberate decision or a later streaming reader.

Parquet reads one bounded batch. CSV/TXT reads at most `LIMIT` rows. Setting `LIMIT=None` opts into full loading and may consume substantially more RAM than the file size. Counts and statistics below refer to loaded rows; exact CSV totals require an explicit full scan.

In [54]:
def json_object(pairs):
    result = dict(pairs)
    if len(result) != len(pairs):
        raise ValueError("Duplicate JSON keys; inspect the original before converting.")
    return result


def structure(value, depth=2):
    if isinstance(value, dict):
        return {key: structure(item, depth - 1) for key, item in islice(value.items(), 5)} if depth else "object"
    if isinstance(value, list):
        return {"length": len(value), "first": [structure(item, depth - 1) for item in value[:2]]} if depth else "array"
    return repr(value)[:100]


def csv_options(path, info):
    header = (None if path.suffix.lower() == ".txt" else 0) if HEADER == "auto" else HEADER
    if header == 0:
        with path.open(encoding=info["encoding"], newline="") as handle:
            for _ in range(SKIP_ROWS):
                handle.readline()
            names = next(csv.reader(handle, delimiter=info["separator"]), [])
        if not names or "" in names or len(names) != len(set(names)):
            raise ValueError("Empty/duplicate column labels. Check HEADER and SKIP_ROWS.")
    return dict(sep=info["separator"], encoding=info["encoding"], skiprows=SKIP_ROWS,
                header=header, usecols=COLUMNS, dtype="string", na_filter=False, index_col=False)


def load_preview(path, info):
    if LIMIT is not None and (not isinstance(LIMIT, int) or LIMIT < 1):
        raise ValueError("LIMIT must be positive or None.")
    if info["format"] == "csv":
        with warnings.catch_warnings():
            warnings.simplefilter("error", pd.errors.ParserWarning)
            return pd.read_csv(path, **csv_options(path, info), nrows=LIMIT), None
    if info["format"] == "parquet":
        if LIMIT is None:
            return pd.read_parquet(path, columns=COLUMNS), None
        with pq.ParquetFile(path) as file:
            batch = next(file.iter_batches(batch_size=LIMIT, columns=COLUMNS), None)
            return (batch.to_pandas() if batch is not None else pd.DataFrame()), None
    if path.stat().st_size > JSON_MAX_BYTES:
        raise ValueError("JSON is too large: inspect the prefix before raising JSON_MAX_BYTES.")
    with path.open(encoding=info["encoding"]) as handle:
        document = json.load(handle, object_pairs_hook=json_object)
    if JSON_PATH is None:
        return None, document
    records = document
    for key in JSON_PATH:
        records = records[key]
    if isinstance(records, dict):
        records = [records]
    if not isinstance(records, list):
        raise ValueError("JSON_PATH must select an object or array.")
    records = records[:LIMIT]
    frame = (pd.DataFrame(records) if all(isinstance(row, dict) for row in records)
             else pd.DataFrame({"value": records}))
    return frame, document

In [55]:
df, document = load_preview(path, info) if path else (None, None)
if document is not None:
    print(json.dumps(structure(document), ensure_ascii=False, indent=2))
    print("Set JSON_PATH after inspecting the structure; nested values remain in cells.")
if df is not None:
    print(f"Loaded subset: {len(df):,} rows, {len(df.columns)} columns")
    display(df.head(SHOW))

# Optional exact CSV count: reads the ENTIRE file, without retaining all chunks.
# with pd.read_csv(path, **csv_options(path, info), chunksize=10_000) as chunks:
#     print(sum(len(chunk) for chunk in chunks))

{
  "length": 11621,
  "first": [
    {
      "Adresse-Lage": "'Am Bondenholz 26, 26-30, 30'",
      "Beschreibung": "'Obsthof Mehrens; 1907; Fabrikation Nordischer Blockhäuser W. Witte; zweigeschossiges Fertig-Holzhau",
      "Bezeichnung": "'Obsthof Mehrens'",
      "Kulturdenkmaltyp": "'Sachgesamtheit'",
      "Schutzumfang": "array"
    },
    {
      "Adresse-Lage": "'Bahnhofstraße 33, 35'",
      "Beschreibung": "'Aktualisierung vorgesehen'",
      "Bezeichnung": "'Kath. Kirche St. Maria - St. Vicelin'",
      "Kulturdenkmaltyp": "'Sachgesamtheit'",
      "Schutzumfang": "array"
    }
  ]
}
Set JSON_PATH after inspecting the structure; nested values remain in cells.
Loaded subset: 5,000 rows, 10 columns


,Adresse-Lage,Beschreibung,Bezeichnung,Kulturdenkmaltyp,Schutzumfang,Gemeinde,Begründung,Objektnummer,Kreis,FotoURL
0,"Am Bondenholz 26, 26-30, 30",Obsthof Mehrens; 1907; Fabrikation Nordischer Blockhäuser W. Witte; zweigesc...,Obsthof Mehrens,Sachgesamtheit,"[Obsthof-Hauptgebäude (Am Bondenholz 26), ehem. Gärtnerhaus (Am Bondenholz 3...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Kulturlandschaftlich]",42758,Stadt Neumünster,NaN
1,"Bahnhofstraße 33, 35",Aktualisierung vorgesehen,Kath. Kirche St. Maria - St. Vicelin,Sachgesamtheit,"[Kath. Kirche St. Maria - St. Vicelin (Bahnhofstraße 33), Pfarrhaus (Bahnhof...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",40602,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/40602.jpg
2,"Bahnhofstraße 40, 42",Tuchfabrik Heinrich Wittorf; 1890-1897; dreigeschossiges Fabrikantenwohnhaus...,Tuchfabrik Heinrich Wittorf,Sachgesamtheit,"[Wohnhaus (Bahnhofstraße 40); Wohnhaus (ehem. Kontor), ehem. Fabrikgebäude (...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",40534,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/40534.jpg
3,"Boostedter Straße 30, 32, Holsatenring 6-8, 10-12, 14",Justizvollzugsanstalt; 1902-1905; Radloff und Saal; Gefängniskomplex in neog...,Justizvollzugsanstalt,Sachgesamtheit,"[Pfortengebäude, Hafthaus A (ehem. Weibergefängnis), Hafthaus C (Vierstrahle...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",15317,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/15317.jpg
4,Brückenstraße,Alteintragung (Aktualisierung vorgesehen),Bahnbetriebswerkstätte Neumünster,Sachgesamtheit,"[Werkstatt- und Verwaltungsgebäude, Lokleitungsgebäude, Ringlokschuppen, Bah...","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Wissenschaftlich]",15133,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/15133.jpg
5,"Buddestraße 7, 9, 11, 13, 15, 17-19, Buddestraße","Eisenbahnersiedlung Neumünster; um 1907, Eisenbahnbauverein; Siedlung nach K...",Eisenbahnersiedlung Neumünster,Sachgesamtheit,"[Wohnhäuser Buddestraße 7, 9, 11, 13, 15, 17-19; Straßenpflasterung Buddestr...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",22591,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/22591.jpg
6,Carlstraße 66,"Landhaus und umgebender Garten; ab 1908, Architekt Hans Roß, Bauherr Alexand...",Villa Moll mit Garten,Sachgesamtheit,"[Villa Moll, Garten]","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",45645,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/45645.jpg
7,Christianstraße,Wasserwerk Neumünster; 1899/1900 u. 1920er; Ensemble bestehnd aus Wasserturm...,Wasserwerk Neumünster,Sachgesamtheit,"[Wasserturm, ehem. Pumpenhaus]","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Technisch]",39919,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/39919.jpg
8,"Christianstraße 11, Am Alten Kirchhof 4, 6, 8-10, Am Alten Kirchhof, Christi...","Anscharkirche; quadratischer, lindengesäumter Kirchhof (alter Friedhof) mit ...",Anscharkirche,Sachgesamtheit,"[Anscharkirche, Alter Friedhof mit historischen Grabmalen (5), Lindenalleenk...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",39579,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/39579.jpg
9,Gartenstraße 10,ehem. Papierfabrik; 1850; 1872; 1911; langgestreckter drei- und viergeschoss...,ehem. Papierfabrik,Sachgesamtheit,"[ehem. Wahlesche Fabrik mit Einfriedung, ehem. Kutscherhaus]","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Technisch]",41798,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/41798.jpg


## 4. Columns and quality

Missing = null or an exact empty string; whitespace and literal `NA` are unchanged.
Nested cells may not support unique/duplicate checks: select scalar columns explicitly.

In [56]:
if df is not None:
    missing = df.isna() | df.map(lambda value: isinstance(value, str) and value == "")
    unique = {}
    for column in df:
        try:
            unique[column] = df[column].nunique(dropna=False)
        except TypeError:
            unique[column] = "nested"
    display(pd.DataFrame({
        "type": df.dtypes, "missing": missing.sum(),
        "missing_%": missing.mean().mul(100), "unique": pd.Series(unique),
    }))
    try:
        print("Repeated rows after their first occurrence:", int(df.duplicated().sum()))
    except TypeError:
        print("Nested cells: use df.duplicated(subset=['your scalar column']).")

,type,missing,missing_%,unique
Adresse-Lage,str,0,0.00,3677
Beschreibung,str,0,0.00,2626
Bezeichnung,str,0,0.00,2736
Kulturdenkmaltyp,str,0,0.00,6
Schutzumfang,object,0,0.00,nested
Gemeinde,str,0,0.00,226
Begründung,object,958,19.16,nested
Objektnummer,int64,0,0.00,5000
Kreis,str,0,0.00,7
FotoURL,str,1249,24.98,3752


Nested cells: use df.duplicated(subset=['your scalar column']).


## 5. Filter, select and sort

Use exact column labels, including umlauts and spaces. Filters below are combined with AND and only search the loaded subset. No automatic municipality matching.

In [57]:
EQUALS = {}           # e.g. {"Ladeeinrichtungs-ID": "1010338"}
SELECT = None        # e.g. ["Ort", "Straße"]
SORT = None          # e.g. "Ort"
FREQUENCY = None     # e.g. "Ort"

view = df
if view is not None:
    for column, value in EQUALS.items():
        view = view.loc[view[column].eq(value)]
    if SORT is not None:
        view = view.sort_values(SORT)
    if SELECT is not None:
        view = view.loc[:, SELECT]
    display(view.head(SHOW))
    if FREQUENCY is not None:
        display(df[FREQUENCY].value_counts(dropna=False).head(SHOW))

# Literal text search, if needed:
# display(df.loc[df["Straße"].str.contains("straße", case=False, regex=False, na=False)].head(SHOW))

,Adresse-Lage,Beschreibung,Bezeichnung,Kulturdenkmaltyp,Schutzumfang,Gemeinde,Begründung,Objektnummer,Kreis,FotoURL
0,"Am Bondenholz 26, 26-30, 30",Obsthof Mehrens; 1907; Fabrikation Nordischer Blockhäuser W. Witte; zweigesc...,Obsthof Mehrens,Sachgesamtheit,"[Obsthof-Hauptgebäude (Am Bondenholz 26), ehem. Gärtnerhaus (Am Bondenholz 3...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Kulturlandschaftlich]",42758,Stadt Neumünster,NaN
1,"Bahnhofstraße 33, 35",Aktualisierung vorgesehen,Kath. Kirche St. Maria - St. Vicelin,Sachgesamtheit,"[Kath. Kirche St. Maria - St. Vicelin (Bahnhofstraße 33), Pfarrhaus (Bahnhof...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",40602,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/40602.jpg
2,"Bahnhofstraße 40, 42",Tuchfabrik Heinrich Wittorf; 1890-1897; dreigeschossiges Fabrikantenwohnhaus...,Tuchfabrik Heinrich Wittorf,Sachgesamtheit,"[Wohnhaus (Bahnhofstraße 40); Wohnhaus (ehem. Kontor), ehem. Fabrikgebäude (...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",40534,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/40534.jpg
3,"Boostedter Straße 30, 32, Holsatenring 6-8, 10-12, 14",Justizvollzugsanstalt; 1902-1905; Radloff und Saal; Gefängniskomplex in neog...,Justizvollzugsanstalt,Sachgesamtheit,"[Pfortengebäude, Hafthaus A (ehem. Weibergefängnis), Hafthaus C (Vierstrahle...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",15317,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/15317.jpg
4,Brückenstraße,Alteintragung (Aktualisierung vorgesehen),Bahnbetriebswerkstätte Neumünster,Sachgesamtheit,"[Werkstatt- und Verwaltungsgebäude, Lokleitungsgebäude, Ringlokschuppen, Bah...","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Wissenschaftlich]",15133,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/15133.jpg
5,"Buddestraße 7, 9, 11, 13, 15, 17-19, Buddestraße","Eisenbahnersiedlung Neumünster; um 1907, Eisenbahnbauverein; Siedlung nach K...",Eisenbahnersiedlung Neumünster,Sachgesamtheit,"[Wohnhäuser Buddestraße 7, 9, 11, 13, 15, 17-19; Straßenpflasterung Buddestr...","Neumünster, Stadt","[Geschichtlich, Städtebaulich]",22591,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/22591.jpg
6,Carlstraße 66,"Landhaus und umgebender Garten; ab 1908, Architekt Hans Roß, Bauherr Alexand...",Villa Moll mit Garten,Sachgesamtheit,"[Villa Moll, Garten]","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",45645,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/45645.jpg
7,Christianstraße,Wasserwerk Neumünster; 1899/1900 u. 1920er; Ensemble bestehnd aus Wasserturm...,Wasserwerk Neumünster,Sachgesamtheit,"[Wasserturm, ehem. Pumpenhaus]","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Technisch]",39919,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/39919.jpg
8,"Christianstraße 11, Am Alten Kirchhof 4, 6, 8-10, Am Alten Kirchhof, Christi...","Anscharkirche; quadratischer, lindengesäumter Kirchhof (alter Friedhof) mit ...",Anscharkirche,Sachgesamtheit,"[Anscharkirche, Alter Friedhof mit historischen Grabmalen (5), Lindenalleenk...","Neumünster, Stadt","[Geschichtlich, Künstlerisch, Städtebaulich]",39579,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/39579.jpg
9,Gartenstraße 10,ehem. Papierfabrik; 1850; 1872; 1911; langgestreckter drei- und viergeschoss...,ehem. Papierfabrik,Sachgesamtheit,"[ehem. Wahlesche Fabrik mit Einfriedung, ehem. Kutscherhaus]","Neumünster, Stadt","[Geschichtlich, Städtebaulich, Technisch]",41798,Stadt Neumünster,https://opendata.schleswig-holstein.de/data/denkmalpflege/fotos/41798.jpg


## 6. Statistics and one quick plot

CSV fields are text. Choose a numeric column explicitly to convert a separate view.
Invalid nonempty values raise an error; IDs and source values remain unchanged.
Without conversion, statistics cover existing numeric JSON/Parquet columns.

In [58]:
NUMBER = None        # e.g. "Nennleistung Ladeeinrichtung [kW]"
DECIMAL = ","

if df is not None:
    numeric = df.select_dtypes(include="number")
    if NUMBER is not None:
        values = df[NUMBER].astype("string").str.replace(DECIMAL, ".", regex=False)
        numeric = pd.to_numeric(values.mask(values.eq("")), errors="raise").to_frame()
    if not numeric.empty:
        display(numeric.describe().T)
        if NUMBER is not None:
            ax = numeric[NUMBER].dropna().plot.hist(bins=30, figsize=(7, 3))
            ax.set(title="Loaded subset", xlabel=NUMBER, ylabel="Records")
            plt.tight_layout()
            plt.show()
            plt.close()

,count,mean,std,min,25%,50%,75%,max
Objektnummer,5000.0,17594.0476,15623.955346,3.0,4904.75,11940.0,26787.5,56509.0


## 7. Optional Schleswig filter

City AGS: **01059075**. Verify the actual source columns first.
Use exact matches, not “contains Schleswig”: city, district and state differ.

Examples: `{"AGS": "01059075"}`, or for the charging register
`{"Ort": "Schleswig", "Postleitzahl": "24837", "Bundesland": "Schleswig-Holstein"}`.
For split municipality keys in headerless files, check the source specification first.
Zero matches in a head preview do not prove absence from the full source.

In [59]:
CITY_FILTER = {}  # Disabled until you configure verified source columns.
if df is not None and CITY_FILTER:
    city = df
    for column, value in CITY_FILTER.items():
        city = city.loc[city[column].eq(value)]
    print(f"City matches in loaded subset: {len(city)}")
    display(city.head(SHOW))

## 8. Notes and future export — inactive

- Source / publisher / license:
- Header, encoding, units, identifiers and geographic coverage:
- Required columns and verified Schleswig matching:
- Quality issues and proposed changes:

Future destination: `data/curated/`, possibly Parquet. **No export code is active.**
Originals and existing curated files remain unchanged. Curated-file versioning is undecided.

References: [pandas CSV options](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [Parquet batch reads](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.ParquetFile.html).
To use Data Wrangler, open the loaded `df` from the notebook.